In [ ]:
from systemflow.hep.utils import hep_graph_from_spreadsheet, hep_with_updated_parameters, hep_construct_graph, dataframes_from_spreadsheet
from systemflow.hep.metrics import Productivity
from systemflow.classifier import get_passed
from systemflow.metrics import precision, recall, f1_score
from systemflow.models import density_scale_model

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
import plotly.graph_objects as go

In [ ]:
# import the data predicting wall time scaling by pileup
scaling = pd.read_excel("wall time scaling.xlsx", sheet_name="Data")
# fit a polynomial to this data for CPU and GPU runtimes
fit_poly = lambda x, k3, k2, k1: k3 * x ** 3 + k2 * x ** 2 + k1 * x
k, cv = curve_fit(fit_poly, scaling["Size"], scaling["Wall Time"])
k_gpu, cv_gpu = curve_fit(fit_poly, scaling["Size"], scaling["Wall Time GPU"])

In [ ]:
# define a dictionary with functions defining the scaling of trigger runtimes with incoming data
funcs = {"Global": lambda x: fit_poly(x, *k), "Intermediate": lambda x: x / 2.0e6}
funcs_gpu = {"Global": lambda x: fit_poly(x, *k_gpu), "Intermediate": lambda x: x / 2.0e6}

In [ ]:
# load data from the spreadsheet which defines the structure of the workflow,
# as well as the parameters for data rates, efficiency, data reduction, and classifier performance
# ...these are taken from predictions for the Run-5 CMS
run5_base = hep_graph_from_spreadsheet("configurations/cms_system_200.xlsx", functions=funcs)

In [ ]:
# read the spreadsheet DataFrames for inspection
det, proc, glob = dataframes_from_spreadsheet("configurations/cms_system_200.xlsx")

In [ ]:
det

In [ ]:
proc

In [ ]:
# calculate the total reduction to keep this constant across experiments
overall_reduction = proc.iloc[4]["Reduction Ratio"] * proc.iloc[5]["Reduction Ratio"]

In [ ]:
overall_reduction

In [ ]:
40e6 / overall_reduction

In [ ]:
"""
Vary the accept rate of the level 1 trigger and inspect its impact on performance and resources required.
v2.0: uses hep_with_updated_parameters to create graph variants without rebuilding from scratch.
"""
def vary_l1t_accept(base_def, l1t_reduction, overall_reduction=5330.0):
    hlt_reduction = overall_reduction / l1t_reduction
    variant = hep_with_updated_parameters(base_def, {
        "Intermediate": {"reduction ratio (1)": l1t_reduction},
        "Global": {"reduction ratio (1)": hlt_reduction}
    })
    result = variant()
    mv = result.metric_values
    power = mv["total power (W)"]
    confusion = mv["pipeline contingency (2x2)"]
    return confusion, power, result

In [ ]:
# the planned Run-5 L1 accept
ex = vary_l1t_accept(run5_base, 53.3)

In [ ]:
ex[2].list_nodes()

In [ ]:
run5_graph = ex[2]

In [ ]:
run5_graph.metric_values["op power (W)"] / density_scale_model(2032) / 1e6

In [ ]:
# its predicted confusion matrix for the workflow
ex[0]

In [ ]:
# and total required power (MW) using 2024's technology
ex[1] / 1e6

In [ ]:
# vary this accept rate from today's rate to the planned Run-5
l1t_reductions = np.linspace(400, 53.3, 101)

In [ ]:
hlt_reductions = overall_reduction / l1t_reductions

In [ ]:
plt.plot(l1t_reductions, hlt_reductions)
plt.xlabel("L1T")
plt.ylabel("HLT")

In [ ]:
run5_res = [vary_l1t_accept(run5_base, r) for r in l1t_reductions]

In [ ]:
def extract_metrics(results):
    all_confusion = np.stack([r[0] for r in results])

    all_power = [r[1] / density_scale_model(2032) for r in results]
    all_power = np.array(all_power)

    all_recall = np.array([recall(all_confusion[i,:,:]) for i in range(all_confusion.shape[0])])
    all_f1 = np.array([f1_score(all_confusion[i,:,:]) for i in range(all_confusion.shape[0])])
    productivity = np.array([np.sum(get_passed(all_confusion[i,:,:])) for i in range(all_confusion.shape[0])])

    metrics = {"confusion": all_confusion,
               "power": all_power,
               "recall": all_recall,
               "f1 score": all_f1,
               "productivity": all_recall * productivity}

    return metrics

In [ ]:
run5_metrics = extract_metrics(run5_res)

In [ ]:
plt.plot(run5_metrics["power"] / 1e6, run5_metrics["f1 score"])
plt.ylabel("F1 Score")
plt.xlabel("DAQ Power (MW)")

In [ ]:
plt.plot(run5_metrics["power"] / 1e6, run5_metrics["productivity"])
plt.ylabel("Retrieved Sample Rate (1/s)")
plt.xlabel("DAQ Power (MW)")

In [ ]:
# repeat this process for alternative configurations
# GPU
run5_gpu_base = hep_graph_from_spreadsheet("configurations/cms_system_200.xlsx", functions=funcs_gpu)
run5_gpu_res = [vary_l1t_accept(run5_gpu_base, r) for r in l1t_reductions]
run5_gpu_metrics = extract_metrics(run5_gpu_res)

In [ ]:
# Smart pixels & GPU
smartpx_base = hep_graph_from_spreadsheet("configurations/cms_system_200_smartpx.xlsx", functions=funcs_gpu)
run5_smartpx_res = [vary_l1t_accept(smartpx_base, r) for r in l1t_reductions]
run5_smartpx_metrics = extract_metrics(run5_smartpx_res)

In [ ]:
# Smart pixels, GPU, & uLED
smartpx_uled_base = hep_graph_from_spreadsheet("configurations/cms_system_200_smartpx_uled.xlsx", functions=funcs_gpu)
run5_smartpx_uled_res = [vary_l1t_accept(smartpx_uled_base, r) for r in l1t_reductions]
run5_smartpx_uled_metrics = extract_metrics(run5_smartpx_uled_res)

In [ ]:
(40e6 / 53.3 / 100)

In [ ]:
40e6 / 53.3

In [ ]:
plt.plot(run5_metrics["power"] / 1e6, run5_metrics["recall"], label = "CPU System")
plt.plot(run5_gpu_metrics["power"] / 1e6, run5_gpu_metrics["recall"], label = "Het. System 1")
plt.plot(run5_smartpx_metrics["power"] / 1e6, run5_smartpx_metrics["recall"], label = "Het. System 2")
#plt.plot(run5_smartpx_uled_metrics["power"] / 1e6, run5_smartpx_uled_metrics["recall"], label = "Het. System 3")
plt.ylabel("Recall")
plt.xlabel("DAQ Power (MW)")
plt.legend()

In [ ]:
fig = go.Figure()
fig.add_scatter(x = run5_metrics["power"] / 1e6, y = run5_metrics["recall"])
#fig.add_scatter(x = run5_gpu_metrics["power"] / 1e6, y = run5_gpu_metrics["recall"], name = "GPU")
#fig.add_scatter(x = run5_smartpx_metrics["power"] / 1e6, y = run5_smartpx_metrics["recall"], name = "GPU & Smart Pixels")

fig.update_xaxes(range=(0, 60))
fig.update_layout(
    title="Recall vs. Power - Phase-1 DAQ",
    xaxis_title="Total Power (MW)",
    yaxis_title="Recall",
    width = 800,
    height = 600,
)

In [ ]:
fig = go.Figure()
fig.add_scatter(x = run5_metrics["power"] / 1e6, y = run5_metrics["recall"], name="CPU")
fig.add_scatter(x = run5_gpu_metrics["power"] / 1e6, y = run5_gpu_metrics["recall"], name = "GPU")
fig.add_scatter(x = run5_smartpx_metrics["power"] / 1e6, y = run5_smartpx_metrics["recall"], name = "GPU & Smart Pixels")

fig.update_xaxes(range=(0, 60))
fig.update_layout(
    title="F1 Score vs. Power",
    xaxis_title="Total Power (MW)",
    yaxis_title="F1 Score",
    legend_title="System",
    width = 800,
    height = 600,
    legend=dict(
        x=0.9,  # x position of the legend (0 corresponds to the left)
        y=0.1,  # y position of the legend (1 corresponds to the top)
        xanchor='right',  # horizontal position anchor ('left', 'center' or 'right')
        yanchor='bottom',  # vertical position anchor ('top', 'middle' or 'bottom')
    )
)

In [ ]:
1.0 - run5_gpu_metrics["power"][-4] / run5_gpu_metrics["power"][-1]

In [ ]:
run5_gpu_metrics["recall"][-1] - run5_gpu_metrics["recall"][-4]

In [ ]:
bar_results = pd.DataFrame({"name": ["CPU", "GPU", "GPU & Data Reduction"],
                            "power": [run5_metrics["power"][-1], run5_gpu_metrics["power"][-1], run5_smartpx_metrics["power"][-1]],
                            "recall": [run5_metrics["recall"][-1], run5_gpu_metrics["recall"][-1], run5_smartpx_metrics["recall"][-1]]})

In [ ]:
fig = go.Figure()

fig.add_bar(x = bar_results["name"], y = bar_results["power"])  # Increase the size and set the font to bold)
fig.update_layout(
    yaxis_title="Total Power (MW)",
    xaxis_title="System",
    title = "Power by System",
    width = 800,
    height = 600,
    title_font = dict(size=36,),
    xaxis=dict(
        title='System',
        titlefont=dict(size=24, family='Arial, bold')  # Bold font for the x-axis title
    ),
    yaxis=dict(
        title='Power Projection (MW)',
        titlefont=dict(size=24, family='Arial, bold')  # Bold font for the y-axis title
    ),
    font = dict(size=18,),
    
)

In [ ]:
fig = go.Figure()
fig.add_scatter(x = run5_metrics["power"][-1:] / 1e6, y = run5_metrics["recall"][-1:], name="CPU")
fig.add_scatter(x = run5_gpu_metrics["power"][-1:] / 1e6, y = run5_gpu_metrics["recall"][-1:], name = "GPU")
fig.add_scatter(x = run5_smartpx_metrics["power"][-1:] / 1e6, y = run5_smartpx_metrics["recall"][-1:], name = "GPU & Smart Pixels")

fig.update_xaxes(range=(0, 60))
fig.update_layout(
    title="Recall vs. Power",
    xaxis_title="Total Power (MW)",
    yaxis_title="Recall",
    legend_title="System",
    width = 800,
    height = 600,
)

In [ ]:
plt.plot(run5_metrics["power"] / 1e6, run5_metrics["recall"], label = "CPU")
plt.plot(run5_gpu_metrics["power"] / 1e6, run5_gpu_metrics["recall"], label = "GPU")
plt.plot(run5_smartpx_metrics["power"] / 1e6, run5_smartpx_metrics["recall"], label = "GPU & Smart Pixels")
#plt.plot(run5_smartpx_uled_metrics["power"] / 1e6, run5_smartpx_uled_metrics["recall"], label = "Het. System 3")
plt.ylabel("Recall")
plt.xlabel("DAQ Power (MW)")
plt.xlim(0, 60)
plt.legend()

In [ ]:
plt.plot(run5_metrics["power"] / 1e6, run5_metrics["productivity"], label = "CPU System")
plt.plot(run5_gpu_metrics["power"] / 1e6, run5_gpu_metrics["productivity"], label = "Het. System 1")
plt.plot(run5_smartpx_metrics["power"] / 1e6, run5_smartpx_metrics["productivity"], label = "Het. System 2")
#plt.plot(run5_smartpx_uled_metrics["power"] / 1e6, run5_smartpx_uled_metrics["recall"], label = "Het. System 3")
plt.ylabel("Retrieved Sample Rate (1/s)")
plt.xlabel("DAQ Power (MW)")
plt.legend()

In [ ]:
plt.plot(run5_metrics["power"] / 1e6, run5_metrics["productivity"], label = "CPU")
plt.plot(run5_gpu_metrics["power"] / 1e6, run5_gpu_metrics["productivity"], label = "GPU")
plt.plot(run5_smartpx_metrics["power"] / 1e6, run5_smartpx_metrics["productivity"], label = "GPU & Smart Px.")
#plt.plot(run5_smartpx_uled_metrics["power"] / 1e6, run5_smartpx_uled_metrics["recall"], label = "Het. System 3")
plt.ylabel("Retrieved Sample Rate (1/s)")
plt.xlabel("DAQ Power (MW)")
plt.xlim(0, 60)
plt.legend()